In [68]:
import os
#os.chdir('../')
print('Working directory:', os.getcwd())

Working directory: /home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk


In [69]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_path: Path
    n_estimators: int
    max_depth: int
    learning_rate: float
    target_column: str


In [70]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.XGBoost
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_path=Path(config.model_path),
            n_estimators=params.n_estimators,
            max_depth=params.max_depth,
            learning_rate=params.learning_rate,
            target_column=schema.name
        )


In [71]:
import os
import pandas as pd
from lightgbm import LGBMClassifier
import joblib
import optuna
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from src.creditrisk.utils import logger


In [72]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_x = pd.read_csv(self.config.train_data_path)
        train_y = pd.read_csv(str(self.config.train_data_path).replace('X_train.csv', 'y_train.csv'))
        train_y_vals = train_y.values.ravel()

        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 80),
                'class_weight': 'balanced',
                'random_state': 42,
                'verbose': -1
            }
            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
            f1_scores = []
            for train_idx, val_idx in cv.split(train_x, train_y_vals):
                X_tr, X_val = train_x.iloc[train_idx], train_x.iloc[val_idx]
                y_tr, y_val = train_y_vals[train_idx], train_y_vals[val_idx]
                model = LGBMClassifier(**params)
                model.fit(X_tr, y_tr)
                preds = model.predict(X_val)
                f1_scores.append(f1_score(y_val, preds))
            return sum(f1_scores) / len(f1_scores)

        logger.info('Starting Optuna optimization for LightGBM (100 trials)...')
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=100)

        best_params = study.best_params
        best_params['class_weight'] = 'balanced'
        best_params['random_state'] = 42
        best_params['verbose'] = -1
        logger.info(f'Best params found by Optuna: {best_params}')

        lgbm = LGBMClassifier(**best_params)
        lgbm.fit(train_x, train_y_vals)

        joblib.dump(lgbm, str(self.config.model_path))
        logger.info(f'Model trained and saved at: {self.config.model_path}')


In [73]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.exception(e)
    raise e


[2026-05-04 19:54:29,115: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-04 19:54:29,116: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-04 19:54:29,118: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-04 19:54:29,119: INFO: common]: created directory at: artifacts
[2026-05-04 19:54:29,120: INFO: common]: created directory at: artifacts/model_trainer
[2026-05-04 19:54:29,223: INFO: 1125609053]: Starting Optuna optimization for LightGBM (100 trials)...
[2026-05-04 19:55:38,808: INFO: 1125609053]: Best params found by Optuna: {'n_estimators': 119, 'max_depth': 3, 'learning_rate': 0.1620884603506881, 'num_leaves': 76, 'class_weight': 'balanced', 'random_state': 42, 'verbose': -1}
[2026-05-04 19:55:39,014: INFO: 1125609053]: Model trained and saved at: artifacts/model_trainer/model.joblib
